# Chapter 7 &mdash; Simulating an NFA: Track the Set of Token Positions

**Concept 5 of the Chapter 7 decomposition:** *Simulating an NFA Without $\varepsilon$: Tracking the Set of Token Positions*

Follow the set of states the tokens occupy; accept when that set meets $F$ and the input is exhausted.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Simulating-Without-Epsilon/Concept-Simulating-Without-Epsilon.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Simulating an NFA is **not** backtracking. Keep a **set** of current states &mdash; where
all live tokens are &mdash; and on each symbol replace it by the union of the $\delta$
images.

$$\text{cur} \ \leftarrow\ \bigcup_{q\in\text{cur}} \delta(q,a)$$

Accept when the input is exhausted and $\text{cur} \cap F \neq \emptyset$.

The cost is $O(|Q|)$ memory and $O(|Q|^2)$ per symbol, **whatever the nondeterminism**
&mdash; and this simulation *is* the subset construction, performed lazily.

## 2. Definitions

### The machine

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> B
B : 0 | 1 -> F
''')

### The set simulation, written out

In [ ]:
def sim(N, s, show=False):
    cur = set(N["Q0"])
    if show: print("start : %s" % sorted(cur))
    for ch in s:
        cur = {t for q in cur for t in step_nfa(N, q, ch)}
        if show: print("on %s : %s" % (ch, sorted(cur)))
    return bool(cur & N["F"])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;4.&nbsp;The Formal NFA: $(Q,\Sigma,\delta,Q_0,F)$ with $\delta: Q\times\Sigma_\varepsilon\to{\cal P}(Q)$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Formal-NFA-Tuple/Concept-Formal-NFA-Tuple.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7-NFA/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;6.&nbsp;$Eclosure$: What $\varepsilon$ Edges Do to Simulation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Eclosure/Concept-Eclosure.ipynb)&nbsp;&rarr;

---

## 3. Tests

The set never exceeds $|Q|$, however many 'paths' there notionally are.

In [ ]:
ok = sim(N, '10101', show=True)
print("\naccepted?", ok, " accepts_nfa says", accepts_nfa(N, '10101'))
assert ok == accepts_nfa(N, '10101')

It agrees with Jove on everything short.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(sim(N, s) == accepts_nfa(N, s) for s in strs)
print("set simulation matches accepts_nfa on all %d strings up to length 10" % len(strs))

Memory is bounded by the state set &mdash; that is the whole point.

In [ ]:
import random
longs = ''.join(random.choice('01') for _ in range(4000))
cur, peak = set(N["Q0"]), 0
for ch in longs:
    cur = {t for q in cur for t in step_nfa(N, q, ch)}
    peak = max(peak, len(cur))
print("input length %d, largest token set ever seen : %d  (|Q| = %d)"
      % (len(longs), peak, len(N["Q"])))
assert peak <= len(N["Q"])

An empty set means every token died &mdash; and it stays empty.

In [ ]:
Dead = md2mc('''NFA
I : 0 -> F
''')
cur = {t for q in Dead["Q0"] for t in step_nfa(Dead, q, '0')}
print("after '0' :", sorted(cur))
cur = {t for q in cur for t in step_nfa(Dead, q, '0')}
print("after '00':", cur, " <- every token has died, and the set stays empty")
assert cur == set() and not accepts_nfa(Dead, '00')

## 4. Animation

The token set as a highlighted group of states, moving as one.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. Why is backtracking simulation exponentially worse than this?
2. Modify `sim` to also report the largest set it ever holds.
3. What is the relationship between `sim` and `nfa2dfa`?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7-NFA/Concept-Simulating-Without-Epsilon')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')